<a href="https://colab.research.google.com/github/zadei/blockermanminesweeperscript/blob/main/v7.0/mania_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## osumapper: create osu! map using Tensorflow and Colab

### -- For osu!mania game mode --

For mappers who don't know how this colaboratory thing works:
- Press Ctrl+Enter in code blocks to run them one by one
- It will ask you to upload .osu file and audio.mp3 after the third block of code
- .osu file needs to have correct timing (you can use [statementreply](https://osu.ppy.sh/users/126198)'s TimingAnlyz tool)
- After uploading them, wait for a few minutes until download pops

Github: https://github.com/kotritrona/osumapper

### Step 1: Installation

First of all, check the Notebook Settings under Edit tab.<br>
Activate GPU to make the training faster.

Then, clone the git repository and install dependencies.

In [15]:
%cd /content/
!rm -rf osumapper
!git clone https://github.com/kotritrona/osumapper.git
%cd osumapper/v7.0
!apt install -y ffmpeg nodejs

with open("requirements_colab.txt", "r") as f:
    reqs = f.read()

reqs = reqs.replace("matplotlib==3.2.2", "matplotlib>=3.5.0")
reqs = reqs.replace("librosa==0.8.0", "librosa>=0.9.0")
reqs = reqs.replace("scikit_learn==0.23.2", "scikit-learn>=1.0.0")
reqs = reqs.replace("scikit-learn==0.23.2", "scikit-learn>=1.0.0")

with open("requirements.txt", "w") as f:
    f.write(reqs)

!cp package_colab.json package.json
!pip install -r requirements.txt
!npm install

/content
Cloning into 'osumapper'...
remote: Enumerating objects: 1292, done.
remote: Counting objects: 100% (248/248), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 1292 (delta 212), reused 194 (delta 194), pack-reused 1044 (from 1)
Receiving objects: 100% (1292/1292), 83.34 MiB | 19.04 MiB/s, done.
Resolving deltas: 100% (718/718), done.
/content/osumapper/v7.0
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
nodejs is already the newest version (18.19.1+dfsg-6ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
⠙npm warn old lockfile
npm warn old lockfile The package-lock.json file was created with an old version of npm,
npm warn old lockfile so supplemental metadata must be fetched from the registry.
npm warn old lockfile
npm warn old lockfile This is a one-time fix-up, please be patient...
npm warn old lockfile
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 5 pac

### Step 2: Choose a pre-trained model
Set the select_model variable to one of:

- "default": default model (choose only after training it)
- "lowkey": model trained with 4-key and 5-key maps (☆2.5-5.5)
- "highkey": model trained with 6-key to 9-key maps (☆2.5-5.5)

In [16]:
from mania_setup_colab import *

select_model = "lowkey"

model_params = load_pretrained_model(select_model);

### Step 3: Upload map and music file<br>
Map file = .osu file with correct timing (**Important:** Set to mania mode and the wished key count!)<br>
Music file = the mp3 file in the osu folder


In [32]:
from google.colab import files
print("Please upload the map file:")
mapfile_upload = files.upload()
for fn in mapfile_upload.keys():
  uploaded_osu_name = fn
  print('Uploaded map file: "{name}" {length} bytes'.format(name=fn, length=len(mapfile_upload[fn])))
print("Please upload the music file:")
music_upload = files.upload()
for fn in music_upload.keys():
  print('Uploaded music file: "{name}" {length} bytes'.format(name=fn, length=len(music_upload[fn])))


Please upload the map file:


Please upload the music file:


### Step 4: Read the map and convert to python readable format



In [24]:
from act_newmap_prep import *
import os

with open(uploaded_osu_name, 'r', encoding='utf-8-sig') as f:
    content = f.read()

sections = {}
current_section = None
lines = content.splitlines()

# Standardize file header
header = "osu file format v14"
for line in lines:
    line_strip = line.strip()
    if not line_strip:
        continue
    if line_strip.startswith("osu file format"):
        continue
    if line_strip.startswith("[") and line_strip.endswith("]"):
        current_section = line_strip
        sections[current_section] = []
    else:
        if current_section is not None:
            sections[current_section].append(line_strip)

# Sanitize [Metadata]
metadata = sections.get("[Metadata]", [])
meta_dict = {}
for item in metadata:
    if ":" in item:
        k, v = item.split(":", 1)
        meta_dict[k.strip()] = v.strip()

# Ensure crucial metadata keys exist
meta_dict.setdefault("Title", "Untitled")
meta_dict.setdefault("Artist", "Unknown")
meta_dict.setdefault("Creator", "Unknown")
meta_dict.setdefault("Version", "Normal")
meta_dict.setdefault("BeatmapID", "0")
meta_dict.setdefault("BeatmapSetID", "-1")

sections["[Metadata]"] = [f"{k}:{v}" for k, v in meta_dict.items()]

# Sanitize [Difficulty]
difficulty = sections.get("[Difficulty]", [])
diff_dict = {}
for item in difficulty:
    if ":" in item:
        k, v = item.split(":", 1)
        diff_dict[k.strip()] = v.strip()

diff_dict.setdefault("HPDrainRate", "5")
diff_dict.setdefault("CircleSize", "4")
diff_dict.setdefault("OverallDifficulty", "5")
diff_dict.setdefault("ApproachRate", "5")

sections["[Difficulty]"] = [f"{k}:{v}" for k, v in diff_dict.items()]

# Reassemble clean file
new_lines = [header]
for section_name, section_lines in sections.items():
    new_lines.append(f"\n{section_name}")
    new_lines.extend(section_lines)
new_lines.append("")

with open(uploaded_osu_name, 'w', encoding='utf-8') as f:
    f.write("\n".join(new_lines))

step4_read_new_map(uploaded_osu_name);

### Step 5: Use model to calculate map rhythm

Parameters:

"note_density" determines how many notes will be placed on the timeline, ranges from 0 to 1.<br>
"hold_favor" determines how the model favors holds against circles, ranges from -1 to 1.<br>
"divisor_favor" determines how the model favors notes to be on X divisors starting from a beat (white, blue, red, blue), ranges from -1 to 1 each.<br>
"hold_max_ticks" determines the max amount of time a hold can hold off, ranges from 1 to +∞.<br>
"hold_min_return" determines the final granularity of the pattern dataset, ranges from 1 to +∞.<br>
"rotate_mode" determines how the patterns from the dataset gets rotated. modes (0,1,2,3,4)
- 0 = no rotation
- 1 = random
- 2 = mirror
- 3 = circulate
- 4 = circulate + mirror

In [28]:
from mania_act_rhythm_calc import *
import tensorflow as tf

# Force load using TensorFlow's legacy saved_model format API to bypass Keras 3 format restrictions
model_path = model_params["rhythm_model"]
try:
    # Try using legacy saving loader
    from tensorflow.python.keras.saving import saved_model_experimental
    model = saved_model_experimental.load(model_path)
except Exception:
    try:
        # Alternate fallback for newer tf packages using the official compat layer
        import tf_keras
        model = tf_keras.models.load_model(model_path)
    except Exception:
        # Fallback to direct low-level loading
        model = tf.saved_model.load(model_path)

npz = step5_load_npz();
params = model_params["rhythm_param"]

predictions = step5_predict_notes(model, npz, params);
notes_each_key = step5_build_pattern(predictions, params, pattern_dataset=model_params["pattern_dataset"]);

3/3 [==============================] - 2s 24ms/step
862 notes predicted.


Do a little modding to the map.

Parameters:

- key_fix: remove continuous notes on single key modes (0,1,2,3) 0=inactive 1=remove late note 2=remove early note 3=divert

In [29]:
modding_params = model_params["modding"]
# modding_params = {
#     "key_fix" : 3
# }

notes_each_key = mania_modding(notes_each_key, modding_params);
notes, key_count = merge_objects_each_key(notes_each_key)

Finally, save the data into an .osu file!

In [30]:
from google.colab import files
from mania_act_final import *

saved_osu_name = step8_save_osu_mania_file(notes, key_count);

files.download(saved_osu_name)

success! built osu map from json data.
elapsed time: 0.023 s

finished on: 2026-10-31 05:41:55.095152


/content/osumapper/v7.0/mania_act_final.py:54: SyntaxWarning: invalid escape sequence '\('
  outname = re.sub("[^a-zA-Z0-9\(\)\[\] \.\,\!\~\`\{\}\-\_\=\+\&\^\@\#\$\%\;\']","", outname);


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
# clean up if you want to make another map!
# colab_clean_up(uploaded_osu_name)

intermediate files cleaned up!


That's it! Now you can try out the AI-created map in osu!.

For bug reports and feedbacks either report it on github or use discord: <br>
[https://discord.com/invite/npmSy7K](https://discord.com/invite/npmSy7K)

<img src="https://i.imgur.com/Ko2wogO.jpg" />